# 초구 예측 — model_03

- 1단계 항목: 24칸 적분에 공분산 반영 (좌우·높이 독립 가정 빼기)
  - 이유: model_02 7장 진단에서 C 치우침(+0.03 / +0.02)의 가장 큰 원인이 독립 가정 (약 0.01)
  - 좌표 NLL에는 이미 공분산을 쓰는데 24칸 확률을 만들 때만 버리고 있었음
- 바꾸는 것: 24칸 계산 방식 + 위치 파라미터(반감기, k_type, k_hand)만 다시 고름
  - 구종은 model_01 값, 위치 기준점은 model_02 값(decay, 반감기 173구) 고정
- 비교 대상: model_02 (2019로 고르고, 2020은 compare 2-1 부트스트랩으로 확인)

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):  # model/ 폴더에서 열면 한 단계 위(데이터·fp_common·results 있는 곳)로
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "model_03"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")

초구 50,709개 (검증 13,458 / 평가 12,615)


## 1. 파라미터

- 구종: model_01 값 고정
- 위치 기준점: model_02 값 고정
- 위치: model_02 값에서 시작 → 아래 3-2에서 다시 고르고 1등으로 덮어씀

In [2]:
# ① 구종 (model_01, 고정): 반감기(공 수), 리그 쪽 당김, 타자 손 쪽 당김
TYPE_HL, TYPE_ALPHA, TYPE_HAND_K = 25.0, 5.48, 5.48
# ② 위치 기준점 (model_02, 고정): 전날까지 구종×손별 반감기 가중, 반감기 173구
LOC_REF, LOC_REF_HL = "decay", 173.0
# ② 위치 (model_02에서 시작, 다시 고름): 반감기(공 수), 투수×구종 / 투수×구종×타자 손 당김
LOC_HL, K_TYPE, K_HAND = 35.4, 548.0, 17.3
# ③ 24칸 적분: 좌우 구간마다 수치 적분 점 개수 (가우스-르장드르, fc.zone_from_gauss_cov 기본값)
ZONE_M = 32

## 2. ① 구종 (model_01과 같음)

In [3]:
def league_type(year):
    return fp.loc[fp["GYEAR"] < year, CLASSES].mean().to_numpy()

def type_online(q, year, half_life, alpha, hand_k):
    S, N = fc.lookup(fc.prior_table(fp, ["pitcher_code"], CLASSES, half_life), q, ["pitcher_code"], CLASSES)
    P = fc.shrink(S, N[:, :1], league_type(year), alpha)
    if hand_k:
        keys = ["pitcher_code", "same_side"]
        S2, N2 = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life), q, keys, CLASSES)
        P = fc.shrink(S2, N2[:, :1], P, hand_k)
    return P

## 3. ② 위치

- 당기는 순서·기준점은 model_02와 같음 (기준점 decay → 투수×구종 → 투수×구종×타자 손)
- 바뀐 것: 정규분포 → 24칸 확률
  - model_02까지 (fc.zone_from_gauss): P(칸) = P(좌우 구간) × P(높이 구간) -> 좌우·높이 상관(공분산) 버림
  - model_03 (fc.zone_from_gauss_cov, model_03에서 만들어 fp_common으로 옮김): P(칸) = ∫ 좌우 구간 P(x) · P(높이 구간 | x) dx
    - 좌우 x를 알면 높이는 다시 정규분포: 중심 = 높이 중심 + ρ·σz·(x 표준화 값), 표준편차 = σz·√(1−ρ²)
    - 적분은 좌우 누적확률 u = Φ((x − 중심)/σx) 위에서 가우스-르장드르 → 바깥 구간(±∞)도 같은 방식, 확률 합은 정확히 1

In [5]:
# 24칸 적분 함수는 fp_common으로 옮김: fc.zone_from_gauss_cov(g, q, m=32) (계산 설명은 위, 코드 주석은 fp_common.py)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

In [6]:
L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 리그 구종×타자 손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 리그 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 리그 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None):  # 위치 모멘트 + 24칸 비율 (n, 29): L2
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    S1, N1 = fc.lookup(fc.prior_table(fp, K1, L_COLS, half_life), q, K1, L_COLS)
    P = fc.shrink(S1, N1, P, k_type)  # 투수×구종 쪽으로 k_type만큼
    S2, N2 = fc.lookup(fc.prior_table(fp, K2, L_COLS, half_life), q, K2, L_COLS)
    return fc.shrink(S2, N2, P, k_hand)  # 투수×구종×손 쪽으로 k_hand만큼

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, cov=True):  # 구종이 t라고 가정했을 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)  # 구종만 t로 바꾼 사본
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl), qt)  # 모멘트 → 정규분포 (높이는 그 타자 존으로 cm 환산)
    return zone_probs(g, qt, cov), g  # 정규분포를 24칸 위에서 적분 (cov=True 공분산 반영)

### 3-1. 적분 확인

- 공분산 0으로 두면 독립 가정 결과(`fc.zone_from_gauss`)와 같아야 함
- 적분 점 개수: 32개 vs 128개 차이가 작아야 함
- 2019 실제 공 300개: 정규분포에서 20만 개씩 뽑아 센 칸 비율과 비교 (표본 오차 약 0.001)

In [7]:
g_val = fc.gauss(loc_online(q_val, fc.VAL_YEAR, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL), q_val)  # 2019 실제 구종 기준 정규분포 (model_02 설정)
P_cov = fc.zone_from_gauss_cov(g_val, q_val)
g0 = g_val[:4] + (np.zeros(len(q_val)),)  # 공분산만 0으로
chk = {"확률 합 − 1 (최대)": np.abs(P_cov.sum(1) - 1).max(),
       "공분산 0 vs 독립 (최대)": np.abs(fc.zone_from_gauss_cov(g0, q_val) - fc.zone_from_gauss(g_val, q_val)).max(),
       "점 32 vs 128 (최대)": np.abs(P_cov - fc.zone_from_gauss_cov(g_val, q_val, m=128)).max()}

# 시뮬레이션: 공 300개, 공마다 2차원 정규분포에서 20만 개 (촐레스키 분해 - 상관 있는 표본)
rng = np.random.default_rng(0)
idx = rng.choice(len(q_val), 300, replace=False)
mx, mz, vx, vz, cv = (a[idx] for a in g_val)
ez = fc.z_edges(q_val["TOP_SZ_CM"].to_numpy()[idx], q_val["BOTTOM_SZ_CM"].to_numpy()[idx])
L21 = cv / np.sqrt(vx); L22 = np.sqrt(np.maximum(vz - L21 ** 2, 1e-9))
err = []
for j in range(len(idx)):
    e1, e2 = rng.standard_normal(200_000), rng.standard_normal(200_000)
    sx_, sz_ = mx[j] + np.sqrt(vx[j]) * e1, mz[j] + L21[j] * e1 + L22[j] * e2
    cell36 = (sz_[:, None] >= ez[j][None, :]).sum(1) * 6 + np.digitize(sx_, fc.X_EDGES)  # 표본마다 36칸 번호
    err.append(np.abs(np.bincount(cell36, minlength=36) / 200_000 @ fc.CELL2ZONE - P_cov[idx[j]]).max())
chk["시뮬레이션 vs 적분 (공별 최대 차이의 평균)"] = np.mean(err)
chk["시뮬레이션 vs 적분 (최대)"] = np.max(err)
chk["|상관계수| 평균"] = np.abs(g_val[4] / np.sqrt(g_val[2] * g_val[3])).mean()
display(pd.Series(chk).round(5))

확률 합 − 1 (최대)                 0.00000
공분산 0 vs 독립 (최대)              0.00000
점 32 vs 128 (최대)              0.00001
시뮬레이션 vs 적분 (공별 최대 차이의 평균)    0.00111
시뮬레이션 vs 적분 (최대)              0.00204
|상관계수| 평균                     0.28357

### 🔍 적분 계산 맞음
- 공분산 0이면 독립 가정 결과와 같음, 확률 합 = 1
- 적분 점 32개 vs 128개 최대 차이 0.00001 -> 32개로 충분
- 시뮬레이션과 공별 최대 차이 평균 0.0011 (표본 오차 수준)
- 좌우·높이 |상관계수| 평균 0.28 -> 버리기엔 큼

### 3-2. 위치 파라미터 다시 고르기 (2019)

- 24칸 계산이 바뀌면 당기는 강도의 최적값도 바뀔 수 있음 -> 반감기, k_type, k_hand 다시 고름 (기준점은 고정)
- 거친 격자는 model_01·02와 같은 후보 → 1등 주변 세밀한 격자
- 기준: 2019 24칸 log loss (실제 구종 기준), H/S/C log loss·좌표 NLL은 참고

In [8]:
def grid_search(score, grid, by="logloss"):  # grid의 모든 조합을 score로 채점 (지표 낮은 순)
    names_ = list(grid)
    combos = pd.MultiIndex.from_product(grid.values(), names=names_).to_frame(index=False)  # 후보의 모든 조합 (한 행 = 한 조합)
    rows = [{**r, **score(**r)} for r in combos.astype(object).where(combos.notna(), None).to_dict("records")]  # NaN → None으로 되돌려 넘김
    return pd.DataFrame(rows).sort_values(by).reset_index(drop=True)

def finer(cands, best, n=3):  
    # 거친 후보 목록에서 best와 양옆 후보 사이를 각각 로그 간격으로 n개씩 나눈 세밀한 후보 (best 가운데, 2n−1개)
    # best가 범위 끝이면 바깥으로 2배 넓힘, best가 None(누적/손 구분 없음)이면 None 그대로"""
    if best is None:
        return [None]
    nums = sorted(c for c in cands if c is not None)  # None 뺀 숫자 후보
    i = nums.index(best)
    lo = nums[i - 1] if i > 0 else best / 2  # 왼쪽 이웃 (없으면 절반)
    hi = nums[i + 1] if i < len(nums) - 1 else best * 2  # 오른쪽 이웃 (없으면 2배)
    vals = np.r_[np.geomspace(lo, best, n), np.geomspace(best, hi, n)[1:]]  # lo~best, best~hi를 같은 비율 간격으로
    return sorted({float(f"{v:.3g}") for v in vals})  # 유효숫자 3자리로 반올림, 중복 제거

def best_of(table, grid):  # 표 1등 행의 파라미터 dict (NaN → None, 숫자는 float)
    r = table.iloc[0]
    return {k: (None if pd.isna(r[k]) else float(r[k])) for k in grid}

def marginal(table, grid, by="logloss"):  # 파라미터마다 값별 가장 좋은 by 지표 (나머지는 그 값에서 가장 좋은 조합) -> 어느 파라미터가 민감한지
    out = {}
    for k in grid:
        s = table.assign(**{k: table[k].astype(object).where(table[k].notna(), "None")}).groupby(k, sort=False)[by].min()
        out[k] = s.reindex([("None" if v is None else v) for v in grid[k] if ("None" if v is None else v) in s.index])
    return out

In [9]:
def score_loc(half_life, k_type, k_hand, cov=True):  # 위치 파라미터 한 조합의 2019 채점: 24칸 log loss·top1·H/S/C + 좌표 NLL (기준점 고정)
    g = fc.gauss(loc_online(q_val, fc.VAL_YEAR, half_life, k_type, k_hand, LOC_REF, LOC_REF_HL), q_val)  # 실제 구종 기준 정규분포
    z = fc.zone_metrics(zone_probs(g, q_val, cov), y_zone[val])
    c = fc.coord_metrics(g, q_val)
    return {"logloss": z["logloss"], "top1": z["top1"], "H/S/C logloss": z["H/S/C logloss"],
            "H/S/C 정확도": z["H/S/C 정확도"], "좌표 NLL": c["NLL"], "평균거리 cm": c["평균거리 cm"]}

GRID_LOC = {"half_life": [None, 1000, 300, 100, 50],  # model_01·02와 같은 거친 격자
            "k_type": [10, 30, 100, 300, 1000, 3000],
            "k_hand": [10, 30, 100, 300, 1000, 3000]}

res = grid_search(score_loc, GRID_LOC)  # 거친 격자
best = best_of(res, GRID_LOC)
grid_f = {k: finer(GRID_LOC[k], best[k]) for k in GRID_LOC}  # 1등 주변 세밀한 후보
res_f = grid_search(score_loc, grid_f)  # 세밀한 격자
print("거친 1등", best, "→ 세밀한 후보", grid_f)
display(res_f.head(5))
for k, s in marginal(res_f, grid_f).items():
    print(k, s.round(4).to_dict())  # 파라미터별 민감도

# 같은 model_02 파라미터에서 계산 방식만 바꾼 효과 / 다시 고른 효과
M02_LOC = (35.4, 548.0, 17.3)  # model_02 위치 파라미터
cmp = pd.DataFrame({"model_02 (독립)": score_loc(*M02_LOC, cov=False),
                    "model_02 파라미터 + 공분산": score_loc(*M02_LOC),
                    "model_03 (다시 고름)": res_f.iloc[0][list(score_loc(*M02_LOC))].to_dict()})
cmp["계산 방식 효과"] = cmp.iloc[:, 1] - cmp.iloc[:, 0]
cmp["다시 고른 효과"] = cmp.iloc[:, 2] - cmp.iloc[:, 1]
display(cmp.astype(float).round(4))

거친 1등 {'half_life': 50.0, 'k_type': 300.0, 'k_hand': 30.0} → 세밀한 후보 {'half_life': [25.0, 35.4, 50.0, 70.7, 100.0], 'k_type': [100.0, 173.0, 300.0, 548.0, 1000.0], 'k_hand': [10.0, 17.3, 30.0, 54.8, 100.0]}


,half_life,k_type,k_hand,logloss,top1,H/S/C logloss,H/S/C 정확도,좌표 NLL,평균거리 cm
0,35.4,1000.0,17.3,2.893256,0.145638,1.072518,0.404741,9.252309,31.817125
1,35.4,548.0,17.3,2.893265,0.145787,1.072520,0.404889,9.252413,31.818392
2,35.4,300.0,17.3,2.893335,0.145564,1.072528,0.405261,9.252674,31.820919
3,25.0,548.0,17.3,2.893497,0.145415,1.072677,0.407267,9.253061,31.807142
4,25.0,1000.0,17.3,2.893512,0.145490,1.072680,0.406821,9.253000,31.806482


half_life {25.0: 2.8935, 35.4: 2.8933, 50.0: 2.8941, 70.7: 2.8956, 100.0: 2.8971}
k_type {100.0: 2.894, 173.0: 2.8935, 300.0: 2.8933, 548.0: 2.8933, 1000.0: 2.8933}
k_hand {10.0: 2.8943, 17.3: 2.8933, 30.0: 2.8943, 54.8: 2.897, 100.0: 2.9007}


,model_02 (독립),model_02 파라미터 + 공분산,model_03 (다시 고름),계산 방식 효과,다시 고른 효과
logloss,2.9222,2.8933,2.8933,-0.0289,-0.0000
top1,0.1456,0.1458,0.1456,0.0002,-0.0001
H/S/C logloss,1.0745,1.0725,1.0725,-0.0020,-0.0000
H/S/C 정확도,0.4039,0.4049,0.4047,0.0010,-0.0001
좌표 NLL,9.2524,9.2524,9.2523,0.0000,-0.0001
평균거리 cm,31.8184,31.8184,31.8171,0.0000,-0.0013


### 🥇1등🥇
#### 반감 35.4, k_type 1000, k_hand 17.3 (기준점 decay 173구 고정)
#### -> 2019 24칸 log loss 2.9222 → 2.8933 (−0.0289)
- 거의 전부 계산 방식 효과 (−0.0289), 다시 고른 효과는 0.0000
  - model_02 파라미터 그대로 둬도 같은 점수 -> 당기는 강도는 24칸 계산 방식과 거의 무관
- k_type: 300–1000에서 2.8933으로 평평 (1000이 세밀한 격자 끝이지만 거친 격자 3000은 이미 짐) -> model_02 548과 사실상 같음
- H/S/C log loss −0.0020 -> 개선의 대부분은 H/S/C가 아닌 같은 그룹 안 어느 칸인지 (대각선 방향 칸)
- 좌표 NLL, 거리는 그대로 (정규분포 자체는 안 바뀜)

In [10]:
# 선택: 2019 24칸 log loss 1등 (실전 예측은 이 값 사용)
b = best_of(res_f, grid_f)
LOC_HL, K_TYPE, K_HAND = b["half_life"], b["k_type"], b["k_hand"]
print(f"선택: half_life={LOC_HL}, k_type={K_TYPE}, k_hand={K_HAND} (기준점 {LOC_REF}, 반감기 {LOC_REF_HL})")

선택: half_life=35.4, k_type=1000.0, k_hand=17.3 (기준점 decay, 반감기 173.0)


### 3-3. 진단: C 치우침

- 예상: 독립 가정 몫(약 0.01)만큼 줄어듦 -> 2019 +0.030 → 약 +0.020, 2020 +0.019 → 약 +0.009
- 열: model_02 (독립) / model_03 (공분산, 다시 고른 파라미터), 행 = 월 + 전체

In [11]:
def hsc_bias(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None, cov=True):  # 월별 H/S/C 예측 평균 − 실제 비율 (실제 구종 기준)
    g = fc.gauss(loc_online(q, year, half_life, k_type, k_hand, ref, ref_hl), q)
    P = zone_probs(g, q, cov) @ fc.GROUP2  # 24칸 확률 → H/S/C (n, 3)
    A = pd.get_dummies(q["zone_group"]).reindex(columns=fc.HSC, fill_value=0).to_numpy(float)  # 실제 H/S/C 원-핫
    d = pd.DataFrame(P - A, columns=["H 차이", "S 차이", "C 차이"])
    d["월"] = q["game_date"].dt.month.to_numpy()
    t = d.groupby("월").mean()
    t.loc["전체"] = d.drop(columns="월").mean()
    t["공수"] = d.groupby("월").size().reindex(t.index).fillna(len(d)).astype(int)
    return t

rows = {}
for year, q in [(fc.VAL_YEAR, q_val), (fc.TEST_YEAR, q_test)]:
    t2 = hsc_bias(q, year, *M02_LOC, LOC_REF, LOC_REF_HL, cov=False)  # model_02
    t3 = hsc_bias(q, year, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL)  # model_03
    rows[(year, "model_02")], rows[(year, "model_03")] = t2["C 차이"], t3["C 차이"]
    print(year, "· model_03 H/S/C 전체", t3.loc["전체", ["H 차이", "S 차이", "C 차이"]].round(3).to_dict())
display(pd.concat(rows, axis=1).round(3))  # 열 = (연도, 모델), 행 = 월 + 전체

2019 · model_03 H/S/C 전체 {'H 차이': -0.01, 'S 차이': -0.01, 'C 차이': 0.02}
2020 · model_03 H/S/C 전체 {'H 차이': -0.003, 'S 차이': -0.007, 'C 차이': 0.009}


2019              2020         
   model_02 model_03 model_02 model_03
월                                     
3     0.045    0.035      NaN      NaN
4     0.034    0.023      NaN      NaN
5     0.034    0.025    0.024    0.013
6     0.036    0.026    0.009   -0.001
7     0.002   -0.008    0.022    0.012
8     0.032    0.022    0.023    0.014
9     0.036    0.027    0.003   -0.005
10   -0.091   -0.098    0.034    0.026
전체    0.030    0.020    0.019    0.009

### 🔍 C 치우침이 예상만큼 줄어듦
#### 전체 C 차이: 2019 0.030 → 0.020, 2020 0.019 → 0.009
- model_02 7장에서 계산한 독립 가정 몫(약 0.01)과 같음, 모든 달에서 0.01 정도씩 줄어듦
- 남은 치우침: 2019 +0.020 = 정규분포 모양 (약 0.006) + 2019 리그 추세 (약 0.014), 2020 +0.009
- H·S 부족도 줄어듦: 2020 H −0.003, S −0.007

## 4. 실전 예측 (예측 구종으로 위치 이어서)

- model_02와 같음, 24칸 확률만 공분산 반영 (`loc_given_type` 기본값 cov=True)

In [12]:
def predict(q, year):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9)
    Pt = type_online(q, year, TYPE_HL, TYPE_ALPHA, TYPE_HAND_K)

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL)
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [13]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR), predict(q_test, fc.TEST_YEAR)], ignore_index=True)
print(pp.shape)

(26073, 48)


## 5. 평가

- 채점은 `fc.evaluate`, model_02와 나란히
- 2020은 확인용 → 2020을 보고 파라미터 다시 고르지 않음

In [14]:
def eval_years(pp_):  # 2019·2020 주요 지표 표
    return pd.DataFrame({"검증 2019": fc.evaluate(pp_, fp, fc.VAL_YEAR), "평가 2020": fc.evaluate(pp_, fp, fc.TEST_YEAR)})

summary = eval_years(pp)
m02_sum = eval_years(pd.read_parquet("results/model_02/per_pitch.parquet"))  # model_02 공별 예측
display(pd.concat({MODEL: summary, "model_02": m02_sum, "차이 (model_02 대비)": summary - m02_sum}, axis=1).round(4))

model_03          model_02          차이 (model_02 대비)        
                     검증 2019  평가 2020  검증 2019  평가 2020          검증 2019 평가 2020
구종 logloss            1.0982   1.0956   1.0982   1.0956           0.0000  0.0000
구종 top1               0.5258   0.5233   0.5258   0.5233           0.0000  0.0000
구종 top2               0.8157   0.8162   0.8157   0.8162           0.0000  0.0000
구종 top3               0.9331   0.9359   0.9331   0.9359           0.0000  0.0000
구종 macro F1           0.2476   0.2857   0.2476   0.2857           0.0000  0.0000
24칸 logloss (실전)      2.9512   2.9479   2.9756   2.9698          -0.0243 -0.0219
24칸 top1              0.1286   0.1330   0.1286   0.1331           0.0000 -0.0001
24칸 top3              0.2873   0.2897   0.2865   0.2859           0.0008  0.0038
H/S/C logloss         1.0753   1.0772   1.0766   1.0778          -0.0013 -0.0007
H/S/C 정확도             0.3935   0.3885   0.3931   0.3874           0.0004  0.0011
24칸 logloss (실제 구종)   2.8933   2.8967   2.9222   2.9225          -0.0289 -0.0259
구종×칸 logloss          3.9914   3.9916   4.0204   4.0177          -0.0289 -0.0261
좌표 평균거리 cm           32.7396  31.8837  32.7399  31.8840          -0.0003 -0.0003
좌표 NLL                9.3195   9.2690   9.3194   9.2689           0.0000  0.0000

- 위치만 바뀜 -> 구종·좌표 지표는 model_02와 같음
- 2020 24칸 log loss 실전 2.9698 → 2.9479 (−0.0219), 실제 구종 2.9225 → 2.8967 (−0.0259)
  - 2019 개선(−0.024 / −0.029)이 2020에도 거의 그대로 이어짐 -> 추세가 아닌 계산 방식 문제였기 때문
  - 실전 개선이 실제 구종 개선의 85% (model_01은 절반) -> 구종을 몰라도 대부분 남는 개선
- 구종×칸 −0.026, 24칸 top3 +0.004, H/S/C log loss −0.0007
- model_01(베이스라인 대비 −0.008)보다 큰 1단계 최대 개선
- **채택 → 다음 비교 출발점**: 2019 1등이고 2020에 나빠진 점수 없음, 다음 모델부터 24칸은 `fc.zone_from_gauss_cov`

## 6. 저장

- `results/model_03/` → `compare.ipynb` 실행하면 자동 포함 (부트스트랩 판정은 compare 2-1에서)

In [15]:
fc.save_results(MODEL, pp, {"summary": summary})

저장: results\model_03 ['per_pitch.parquet', 'summary.csv']
